In [ ]:
slice_ensemble = {
                    'NAME_STUDENT': 'worthy-waterfall_2',
                    'ID_NUM': 'splendid-totem-15',
                    'STREET_ADDRESS': 'valiant-water_1',
                    'URL_PERSONAL': 'splendid-totem-15',
                    'PHONE_NUM': 'swift-feather',
                    'USERNAME': 'splendid-totem-15_1',
                    'EMAIL': 'splendid-totem-15'
                }

model_paths = {
                'worthy-waterfall_2': '/kaggle/input/pii-detect-deberta3large-models/worthy-waterfall-chk-3200-f1_0.9590',
                'splendid-totem-15': '/kaggle/input/pii-detect-deberta3large-models/splendid-totem-15-checkpoint-1800-f1_0.9659',
                'swift-feather': '/kaggle/input/pii-detect-deberta3large-models/swift-feather-checkpoint-950-f1_0.9340',
                'valiant-water_1': '/kaggle/input/pii-detect-deberta3large-models/valiant-water_1-chk-1200-f1_0.9479',
                'splendid-totem-15_1': '/kaggle/input/pii-detect-deberta3large-models/splendid-totem-15_1-chk-2100-f1_0.9572',
                }

threshold = 0.99

INFERENCE_MAX_LENGTH = 3500

# Load libraries

In [ ]:
import os
import re
import gc

import json
import argparse
from itertools import chain
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments, DataCollatorForTokenClassification
from datasets import Dataset
import numpy as np

# NLP Tokenization and Model Preparation

In [ ]:


# Function definition for 'tokenize'
def tokenize(example, tokenizer):
    # Initializing two lists: 'text' for storing tokens and 'token_map' for mapping tokens to their original positions.
    text = []
    token_map = []
    
    # Starting index at 0 to track tokens.
    idx = 0
    
    # Iterating through tokens and their associated trailing whitespaces.
    for t, ws in zip(example["tokens"], example["trailing_whitespace"]):
        
        # Adding each token to the 'text' list.
        text.append(t)
        
        # Extending 'token_map' with the current index repeated as many times as the length of the token.
        token_map.extend([idx] * len(t))
        
        # Adding a space for trailing whitespace and marking it with '-1' in 'token_map'.
        if ws:
            text.append(" ")
            token_map.append(-1)
            
        # Incrementing 'idx' for the next token.
        idx += 1
        
    # Tokenizing the concatenated 'text' and returning offset mappings along with 'token_map'.
    tokenized = tokenizer("".join(text), return_offsets_mapping=True, truncation=False, max_length=INFERENCE_MAX_LENGTH)
    
    # Returning a dictionary containing the tokenized data and the 'token_map'.
    return {
        **tokenized,
        "token_map": token_map,
    }


In [ ]:
data = json.load(open("/kaggle/input/pii-detection-removal-from-educational-data/test.json"))

# Create a dataset from the loaded data
ds = Dataset.from_dict({
    "full_text": [x["full_text"] for x in data],
    "document": [x["document"] for x in data],
    "tokens": [x["tokens"] for x in data],
    "trailing_whitespace": [x["trailing_whitespace"] for x in data],
})

# Initialize a tokenizer and model from the pretrained model path
# model_paths = {'/kaggle/input/pii-deberta-models/cola-de-piiranha' : 2/10,
#               '/kaggle/input/pii-deberta-models/cuerpo-de-piiranha' : 2/10,
#               '/kaggle/input/pii-deberta-models/cabeza-de-piiranha' : 2/10,
#               '/kaggle/input/pii-deberta-models/cabeza-del-piinguuino' : 5/10}


first_model_path = list(model_paths.values())[0]

tokenizer = AutoTokenizer.from_pretrained(first_model_path)

# Tokenize the dataset using the 'tokenize' function in parallel
ds = ds.map(tokenize, fn_kwargs={"tokenizer": tokenizer}, num_proc = 2)


#display(Image(filename='/kaggle/input/pii-deberta-models/piiratefisk.png'))



In [ ]:
train_data = json.load(open("/kaggle/input/pii-detection-removal-from-educational-data/train.json"))
all_labels = sorted(list(set(chain(*[x["labels"] for x in train_data]))))
label2id = {l: i for i,l in enumerate(all_labels)}
id2label = {v:k for k,v in label2id.items()}

target = [
    'B-EMAIL', 'B-ID_NUM', 'B-NAME_STUDENT', 'B-PHONE_NUM',
    'B-STREET_ADDRESS', 'B-URL_PERSONAL', 'B-USERNAME', 'I-ID_NUM',
    'I-NAME_STUDENT', 'I-PHONE_NUM', 'I-STREET_ADDRESS', 'I-URL_PERSONAL'
]

print(id2label)

del train_data
_ = gc.collect()

# Inference

In [ ]:
import gc
import torch
import numpy as np

from scipy.special import softmax


all_preds = []

# Calculate the total weight
# total_weight = sum(model_paths.values())

# Directory for saving intermediate predictions
# intermediate_dir = './intermediate_predictions'
# os.makedirs(intermediate_dir, exist_ok=True)

preds_dict = dict()

for idx, (model_name, model_path) in enumerate(model_paths.items()):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    collator = DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=16)
    args = TrainingArguments(
        ".",
        per_device_eval_batch_size=1,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        data_collator=collator,
        tokenizer=tokenizer,
    )
    predictions = trainer.predict(ds).predictions
    predictions = softmax(predictions, axis=-1)
    
    preds_dict[model_name] = predictions
    
    # Save weighted_predictions to disk
#     np.save(os.path.join(intermediate_dir, f'weighted_preds_{idx}.npy'), weighted_predictions)
    
    # Clear memory
    del model, trainer, tokenizer, predictions
    torch.cuda.empty_cache()
    gc.collect()

# # Initialize an array for aggregated predictions
# aggregated_predictions = None

# # Load and aggregate predictions
# for file_name in os.listdir(intermediate_dir):
#     weighted_predictions = np.load(os.path.join(intermediate_dir, file_name))
#     if aggregated_predictions is None:
#         aggregated_predictions = weighted_predictions
#     else:
#         aggregated_predictions += weighted_predictions

# # Finally, compute the weighted average of predictions
# weighted_average_predictions = aggregated_predictions / total_weight


In [ ]:
preds_dict.keys()

In [ ]:
# Create a new array for final predictions, initialized to zero
final_predictions = np.zeros_like(next(iter(preds_dict.values())))  # This assumes all arrays are the same shape
base_model = 'splendid-totem-15'

# For each PII type, use the model's predictions specified in slice_ensemble
for pii_type, model in slice_ensemble.items():
    # Begin and Inside tags for each PII type
    b_tag = 'B-' + pii_type
    i_tag = 'I-' + pii_type

    # Extract predictions using the appropriate model and assign them to the final predictions array
    if b_tag in label2id:
        final_predictions[..., label2id[b_tag]] = preds_dict[model][..., label2id[b_tag]]
    if i_tag in label2id:
        final_predictions[..., label2id[i_tag]] = preds_dict[model][..., label2id[i_tag]]
    

final_predictions[..., label2id['O']] = preds_dict[base_model][..., label2id['O']]

# b_tag = 'B-PHONE_NUM'
# i_tag = 'I-PHONE_NUM'
# final_predictions[..., label2id[b_tag]] = (preds_dict['valiant-water'][..., label2id[b_tag]] + preds_dict['swift-feather'][..., label2id[b_tag]]) / 2
# final_predictions[..., label2id[i_tag]] = (preds_dict['valiant-water'][..., label2id[b_tag]] + preds_dict['swift-feather'][..., label2id[b_tag]]) / 2

final_predictions.shape

In [ ]:
config = json.load(open(Path(model_path) / "config.json"))
id2label = config["id2label"]
preds = final_predictions.argmax(-1)
preds_without_O = final_predictions[:,:,:12].argmax(-1)
O_preds = final_predictions[:,:,12]



preds_final = np.where(O_preds < threshold, preds_without_O , preds)

# Postproc

In [ ]:
triplets = []
pairs = set()  # membership operation using set is faster O(1) than that of list O(n)

processed = []

# For each prediction, token mapping, offsets, tokens, and document in the dataset
for p, token_map, offsets, tokens, doc in zip(preds_final, ds["token_map"], ds["offset_mapping"], ds["tokens"], ds["document"]):

    # Iterate through each token prediction and its corresponding offsets
    for token_pred, (start_idx, end_idx) in zip(p, offsets):
        label_pred = id2label[str(token_pred)]  # Predicted label from token

        # If start and end indices sum to zero, continue to the next iteration
        if start_idx + end_idx == 0:
            continue

        # If the token mapping at the start index is -1, increment start index
        if token_map[start_idx] == -1:
            start_idx += 1

        # Ignore leading whitespace tokens ("\n\n")
        while start_idx < len(token_map) and tokens[token_map[start_idx]].isspace():
            start_idx += 1

        # If start index exceeds the length of token mapping, break the loop
        if start_idx >= len(token_map):
            break

        token_id = token_map[start_idx]  # Token ID at start index

        # Ignore "O" predictions and whitespace tokens
        if label_pred in ("O", "B-EMAIL", "B-PHONE_NUM", "I-PHONE_NUM") or token_id == -1:
            continue

        pair = (doc, token_id)

        if pair not in pairs:
            processed.append({"document": doc, "token": token_id, "label": label_pred, "token_str": tokens[token_id]})
            pairs.add(pair)
# We've gathered the valuable triplets from the dataset, ready for analysis!


In [ ]:
from spacy.lang.en import English
nlp = English()

def find_span(target: list[str], document: list[str]) -> list[list[int]]:
    idx = 0
    spans = []
    span = []

    for i, token in enumerate(document):
        if token != target[idx]:
            idx = 0
            span = []
            continue
        span.append(i)
        idx += 1
        if idx == len(target):
            spans.append(span)
            span = []
            idx = 0
            continue
    
    return spans

In [ ]:
data = json.load(open("/kaggle/input/pii-detection-removal-from-educational-data/test.json"))

email_regex = re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+')
phone_num_regex = re.compile(r"(\(\d{3}\)\d{3}\-\d{4}\w*|\d{3}\.\d{3}\.\d{4})\s")
emails = []
phone_nums = []

for _data in data:
    # email
    for token_idx, token in enumerate(_data["tokens"]):
        if re.fullmatch(email_regex, token) is not None:
            emails.append(
                {"document": _data["document"], "token": token_idx, "label": "B-EMAIL", "token_str": token}
            )
    # phone number
    matches = phone_num_regex.findall(_data["full_text"])
    if not matches:
        continue
    for match in matches:
        target = [t.text for t in nlp.tokenizer(match)]
        matched_spans = find_span(target, _data["tokens"])
    for matched_span in matched_spans:
        for intermediate, token_idx in enumerate(matched_span):
            prefix = "I" if intermediate else "B"
            phone_nums.append(
                {"document": _data["document"], "token": token_idx, "label": f"{prefix}-PHONE_NUM", "token_str": _data["tokens"][token_idx]}
            )

# Submission

In [ ]:
df = pd.DataFrame(processed + phone_nums + emails)

# Assign each row a unique 'row_id'
df["row_id"] = list(range(len(df)))

# Display a glimpse of the first 100 rows of your data
display(df.head(100))

# Cast your findings into a CSV file for further exploration
df[["row_id", "document", "token", "label"]].to_csv("submission.csv", index=False)

# May the winds of fortune guide ye to untold discoveries!
